# Layer 2 is three numbers

The run from **`train_three_params.py`**. Each block is pushed to a single degree of freedom and
the half the softmax cannot see is nailed to zero.

| block | before (`train_l1_two_parts.py`) | here |
|---|---|---|
| L1 $W_{qk}$[0:8, 0:8] | two column halves, 64 cells -> 2 dof | signal cols **frozen 0**, noise cols one value -> **1 dof**, 32 cells |
| L1 $W_{qk}$[8:108, 8:108] | every cell free, 9,900 | unchanged, 9,900 |
| L2 $W_{qk}$[0:8, 0:8] (A) | two halves, 64 cells -> 2 dof | signal cols **frozen 0**, noise cols one value -> **1 dof**, 32 cells |
| L2 $W_{qk}$[0:4, 108:112] (B) | diag + off-diag -> 2 dof | **diagonal only**, one value -> **1 dof**, 4 cells; off-diagonal deleted |
| L2 $W_{qk}$[108:112, 108:112] (C) | diag + off-diag -> 2 dof | **diagonal only**, one value -> **1 dof**, 4 cells; off-diagonal deleted |
| | 10,060 cells / 9,908 dof | **9,972 cells / 9,904 dof** |

**Layer 2 is now three numbers**: $a_2$ (noise veto), $b_2$ (induction), $c_2$ (state).

**Why freezing half of each block costs nothing.** $\mathrm{softmax}(s + \text{const}) =
\mathrm{softmax}(s)$, so adding one number to a query row across a complete group of keys
changes no output. For the A blocks the key group is the key's token one-hot, which sums to
exactly 1, so $(\text{sig } +a,\ \text{noi } -a)$ and $(\text{sig } 0,\ \text{noi } -2a)$ are
the same circuit. For B and C the group is layer 1's output on its 4 signal slots, which sums to
$\approx 0.998$, so it is a near-gauge rather than an exact one (measured cost of a shift on
that block: under 0.01 nats -- CLAUDE.md section 8.7).

So this is a reparametrisation, not a weaker model. The point is that it **pins the gauge**, so
the learned numbers can be read straight off and lined up against the construction's $a, b, c$.

**Storage.** Snapshots now hold one value per degree of freedom, not one per cell -- 3 numbers
a step for layer 2 instead of 40. Rebuild with
`W = zeros(shape); W[mask] = row[dofidx]`, and `load_dof()` below does it for you.

In [ ]:
import os, json, glob, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import HTML, display
plt.rcParams['animation.embed_limit'] = 300

V, Vs, T = 8, 4, 100
D1, D2 = V + T, 2 * (V + T)          # 108, 216

RUN_PREFIX = 'tied4_'          # train_three_params.py tags its runs this way;
                               # train.py uses 'tied_', train_l1_two_parts.py 'tied2_'

def discover_run():
    '''the RUN_PREFIX run with the most epochs that has snapshots -- falls back to the newest'''
    cands = [d for d in glob.glob(f'runs/{RUN_PREFIX}*/')
             if glob.glob(os.path.join(d, 'snapmeta_*.json'))]
    if not cands:
        raise FileNotFoundError(
            f'no runs/{RUN_PREFIX}* has snapshots yet -- start train_three_params.py first.\n'
            f'  (runs present: {sorted(os.path.basename(d.rstrip("/")) for d in glob.glob("runs/*/"))})')
    def eps(d):
        m = re.search(r'_ep-(\d+)', d)
        return int(m.group(1)) if m else -1
    return max(cands, key=lambda d: (eps(d), os.path.getmtime(d))).rstrip('/')

# Pin the run by name. Several tied4_ runs exist (adam ep-100, adam ep-200, an sgd probe at
# lr 1e4) so auto-discovery cannot choose. Set RUN_NAME = None to fall back to
# "the RUN_PREFIX run with the most epochs".
RUN_NAME = 'tied4_opt-adam_lr-1.0_init-zeros_wov-fixed_blocks-relevant_readout-bias_ep-100'

if RUN_NAME:
    RUN = os.path.join('runs', RUN_NAME)
    if not glob.glob(f'{RUN}/snapmeta_*.json'):
        raise FileNotFoundError(
            f'{RUN} has no snapshots.\n  runs present: '
            f'{sorted(os.path.basename(d.rstrip("/")) for d in glob.glob("runs/*/"))}')
else:
    RUN = discover_run()
print(RUN, '\n')

def thresholds_present():
    '''read them off the files -- a run may use any --thresholds, not just 0.6/0.8/1.0'''
    out = []
    for p in glob.glob(f'{RUN}/snapmeta_x_*.json'):
        m = re.search(r'snapmeta_x_([0-9.]+)\.json$', p)
        if m:
            out.append(float(m.group(1)))
    return sorted(out)

THRS = thresholds_present()
if not THRS:
    raise FileNotFoundError(f'{RUN} has no snapmeta_x_*.json')
for _t in THRS:
    _m = json.load(open(f'{RUN}/snapmeta_x_{_t}.json'))
    _done = _m['w_rows_written'] >= _m['rows_allocated']
    print(f"  alpha={_t}: w={_m['w_rows_written']}/{_m['rows_allocated']}  g={_m['g_rows_written']}  "
          f"{'complete' if _done else 'STILL TRAINING -- cells below use what exists'}")

def aligned_losses(thr):
    '''train and test loss indexed by the weights they describe.

    train.py already evaluates BOTH after the step, so no shift is needed here -- unlike
    folder 18, where train_loss was the pre-step value and lagged test by one. `train_loss_pre`
    is kept in the metrics for exactly that comparison.

    Empty arrays come back when a threshold has not finished: metrics_*.json is written when
    the threshold ENDS, while snapmeta / w / g appear from step 0. Callers check .size.
    '''
    p = f'{RUN}/metrics_seed0_x_{thr}.json'
    if not os.path.exists(p):
        e = np.array([])
        return e, e, e, float('nan'), 0
    h = json.load(open(p))['history']
    return (np.asarray(h['step']), np.asarray(h['train_loss']), np.asarray(h['test_loss']),
            h['test_loss'][-1], h['step'][-1])

def load_dof(param, thr, kind='w'):
    '''(mask, dofidx, rows) for the new per-dof storage.

    A stored row holds ONE VALUE PER DEGREE OF FREEDOM, not one per cell -- layer 2 is three
    numbers. Rebuild any step k with   W = zeros(shape); W[mask] = rows[k][dofidx].
    '''
    mask = np.load(f'{RUN}/mask_{param}_x_{thr}.npy')
    dofidx = np.load(f'{RUN}/dofidx_{param}_x_{thr}.npy')
    rows = np.load(f'{RUN}/{kind}_{param}_x_{thr}.npy', mmap_mode='r')
    return mask, dofidx, rows

def group_map(meta, param, mask, dofidx):
    '''full-shape int array: which dof each cell belongs to.

    -1 means "not worth outlining": a dof that owns a single cell (every cell of the position
    block). Only dofs shared by two or more cells get a black outline in the panels.
    '''
    gid = np.full(meta['shapes'][param], -1, dtype=int)
    gid[mask] = dofidx
    cnt = np.bincount(dofidx, minlength=int(dofidx.max()) + 1)
    solo = np.flatnonzero(cnt <= 1)
    if solo.size:
        gid[np.isin(gid, solo)] = -1
    return gid

def panel(ax, A, title, vmax, annot, gid=None):
    v = vmax if vmax > 0 else 1.0
    ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
              norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    if annot:
        lab = [['0' if z == 0 else (f'{z:.3g}' if abs(z) >= 0.01 else f'{z:.1e}') for z in row]
               for row in A]
        wid = max(len(x) for row in lab for x in row)
        fs = (12.5 if A.shape[0] <= Vs else 6.4) * min(1.0, 5.0 / wid)
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, lab[i][j], ha='center', va='center', fontsize=fs,
                        color='white' if abs(A[i, j]) > v / 3 else 'black')
        if A.shape[0] <= Vs:
            ax.set_xticks(range(A.shape[1])); ax.set_yticks(range(A.shape[0]))
            ax.set_xticklabels([f'k{q}' for q in range(A.shape[1])], fontsize=8)
            ax.set_yticklabels([f'q{q}' for q in range(A.shape[0])], fontsize=8)
            ax.tick_params(length=0)
        else:
            ax.set_xticks([]); ax.set_yticks([])
    else:
        ax.set_xticks([]); ax.set_yticks([])
    if gid is not None:                       # outline the tie groups: one parameter each
        for i in range(gid.shape[0]):
            for j in range(gid.shape[1]):
                if j + 1 < gid.shape[1] and gid[i, j] != gid[i, j + 1]:
                    ax.plot([j + .5, j + .5], [i - .5, i + .5], color='k', lw=2.2)
                if i + 1 < gid.shape[0] and gid[i, j] != gid[i + 1, j]:
                    ax.plot([j - .5, j + .5], [i + .5, i + .5], color='k', lw=2.2)
    ax.set_title(f'{title}\n$\\|\\cdot\\|_F$ {np.linalg.norm(A):.4g}', fontsize=9)

## The animation

`ANIM_FROM` / `ANIM_TO` are epochs, both inclusive. Colour scales are fixed across the frames
*shown*, so a late window is scaled to itself — which is what makes small late changes visible,
and also means two windows are not comparable by colour.

In [ ]:
ANIM_THR  = 0.6      # any alpha listed above
ANIM_FROM = 0        # first epoch  (inclusive)
ANIM_TO   = 50       # last  epoch  (inclusive)
#   51 frames of this 6-panel figure is ~17 MB inline. The cell prints the size; shorten the
#   range or lower ANIM_DPI if the notebook gets heavy.
LOSS_ROW_H = 1.24     # height of the loss row, in units of one matrix row (0.62 = old)
ANIM_DPI   = 88       # six panels wide now -- drop this if the notebook feels heavy

# A 10x10 window from the MIDDLE of layer 1's 100x100 position block, blown up to a panel
# of its own and placed third, next to the full block. The full 100x100 is 9,900 near-identical
# ramp rows in which nothing is legible; this window is where the ramp actually lives -- it
# straddles the causal diagonal, so the decay with distance is visible cell by cell.
ZOOM_AT, ZOOM_N = 45, 10                      # position block rows/cols [45:55]
ZOOM = slice(V + ZOOM_AT, V + ZOOM_AT + ZOOM_N)

# the six panels: (param, row slice, col slice, label, annotate?)
CUT = [('layer1.W_qk', slice(0, V),        slice(0, V),        'L1 tok x tok',     True),
       ('layer1.W_qk', slice(V, D1),       slice(V, D1),       'L1 pos x pos',     False),
       ('layer1.W_qk', ZOOM,               ZOOM,
        f'L1 pos x pos  [{ZOOM_AT}:{ZOOM_AT + ZOOM_N}]  zoom',                     True),
       ('layer2.W_qk', slice(0, V),        slice(0, V),        'L2 A  noise suppr.', True),
       ('layer2.W_qk', slice(0, Vs),       slice(D1, D1 + Vs), 'L2 B  induction',  True),
       ('layer2.W_qk', slice(D1, D1 + Vs), slice(D1, D1 + Vs), 'L2 C  state',      True)]

def animate(thr, first=0, last=50, interval=450):
    mp = f'{RUN}/snapmeta_x_{thr}.json'
    if not os.path.exists(mp):
        raise FileNotFoundError(f'no snapshots for alpha={thr} in {RUN}')
    mt = json.load(open(mp))
    kmax = min(mt['w_rows_written'] - 1, mt['g_rows_written'] - 1)   # step k needs w[k] AND g[k]
    a, b = max(0, int(first)), min(int(last), kmax)
    if b < a:
        raise ValueError(f'empty range [{first}, {last}] -> [{a}, {b}]; '
                         f'this run has steps 0..{kmax} for alpha={thr}')
    if b < int(last):
        print(f'alpha={thr}: only steps 0..{kmax} recorded so far -> showing [{a}, {b}]')
    n = b - a + 1
    print(f'alpha={thr}: epochs {a}..{b}   ({n} frames)')

    Mk, DOF, Wf, Gf = {}, {}, {}, {}
    for p in mt['params']:
        Mk[p], DOF[p], rw = load_dof(p, thr, 'w')
        Wf[p] = np.asarray(rw[a:b + 1])
        Gf[p] = np.asarray(load_dof(p, thr, 'g')[2][a:b + 1])
    GID = {p: group_map(mt, p, Mk[p], DOF[p]) for p in mt['params']}

    def blocks(src, p, rs, cs):
        '''rebuild the block: one stored value per dof, broadcast to every cell of that dof'''
        out = np.empty((n, rs.stop - rs.start, cs.stop - cs.start))
        for k in range(n):
            A = np.zeros(mt['shapes'][p]); A[Mk[p]] = src[p][k][DOF[p]]; out[k] = A[rs, cs]
        return out

    Ws = [blocks(Wf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    Gs = [blocks(Gf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    vw = [float(np.abs(x).max()) for x in Ws]
    vg = [float(np.abs(x).max()) for x in Gs]
    gids = [GID[p][rs, cs] if an else None for p, rs, cs, _, an in CUT]

    st, tr, te, fin, nlast = aligned_losses(thr)
    st, tr, te = st[a:b + 1], tr[a:b + 1], te[a:b + 1]

    nb_ = len(CUT)
    # Three rows. The loss curve is no longer a narrow column on the right: it gets its own
    # row underneath both matrix rows, stretched across the full width, with the same panel
    # height it had before (LOSS_ROW_H is in units of one matrix row).
    f = plt.figure(figsize=(4.6 * nb_, 9.2 + 4.6 * LOSS_ROW_H + 1.55), dpi=ANIM_DPI)
    g = f.add_gridspec(3, nb_, height_ratios=[1, 1, LOSS_ROW_H], wspace=0.10, hspace=0.16)
    ax = np.array([[f.add_subplot(g[r, c_]) for c_ in range(nb_)] for r in range(2)])
    al = f.add_subplot(g[2, :])
    f.subplots_adjust(top=0.92, bottom=0.06, left=0.025, right=0.99)

    def dr(i_):
        ep = a + i_
        for a_ in ax.ravel():
            a_.clear()
        for j_, (_, _, _, ttl, an) in enumerate(CUT):
            panel(ax[0][j_], Ws[j_][i_], f'W  {ttl}', vw[j_], an, gids[j_])
            panel(ax[1][j_], Gs[j_][i_], f'grad  {ttl}', vg[j_], an, gids[j_])
        al.clear()
        if not st.size:                      # threshold still running: no metrics file yet
            al.text(0.5, 0.5, f'metrics_seed0_x_{thr}.json not written yet\n'
                              f'(it appears when this threshold finishes)',
                    transform=al.transAxes, ha='center', va='center',
                    fontsize=12, color='0.45')
            al.set_xticks([]); al.set_yticks([])
            f.suptitle(f'$\\alpha$={thr}   epoch {ep}   (range {a}..{b})   '
                       f'top: $W$;  middle: $g[{ep}]$   '
                       f'[black outlines = one tied parameter]', fontsize=13)
            return []
        al.plot(st, tr, color='#1f77b4', lw=1.8, label='train')
        al.plot(st, te, color='#ff7f0e', lw=1.4, ls='--', label='test')
        al.axvline(ep, color='k', lw=1.1, alpha=0.65)
        al.plot([ep], [tr[i_]], 'o', color='#1f77b4', ms=8, zorder=5)
        al.plot([ep], [te[i_]], 'o', color='#ff7f0e', ms=8, zorder=5)
        al.annotate(f'epoch {ep}\ntrain {tr[i_]:.5f}\ntest  {te[i_]:.5f}',
                    xy=(0.96, 0.95), xycoords='axes fraction', ha='right', va='top',
                    fontsize=8.5, family='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='0.7', alpha=0.9))
        al.set_xlim(a - 0.5, b + 0.5)
        al.set_xlabel('optimizer step (= epoch)', fontsize=9)
        al.set_ylabel('cross entropy', fontsize=9); al.tick_params(labelsize=8)
        al.set_title(f'loss over epochs {a}..{b}\n(run ends at {fin:.5f} after {nlast})',
                     fontsize=9)
        al.grid(alpha=0.3); al.legend(fontsize=8, loc='lower left')
        note = '   --  zero init: W all zero, and layer 1 has NO gradient yet' if ep == 0 else ''
        f.suptitle(f'$\\alpha$={thr}   epoch {ep}   (range {a}..{b})   '
                   f'top: $W$ before update {ep+1};  middle: $g[{ep}]$ at those weights;  '
                   f'bottom: the loss'
                   f'   [black outlines = one tied parameter]{note}', fontsize=13)
        return []

    an_ = animation.FuncAnimation(f, dr, frames=n, interval=interval, blit=False)
    plt.close(f)
    html = an_.to_jshtml()
    print(f'  animation {len(html) / 1e6:.1f} MB   ({nb_} panels x 2 rows + loss row)')
    return HTML(html)

display(animate(ANIM_THR, ANIM_FROM, ANIM_TO))

## The three numbers, on their own

Same data as the L2 panels above, read as scalars in logit units.

In [ ]:
# ---------------------------------------------------------- the three layer-2 numbers
# The headline of this run: layer 2's whole W_qk is a2 (noise veto), b2 (induction), c2 (state).
# Read them in LOGIT units -- the stored entry divided by sqrt(108), which is what the softmax
# sees -- and watch them form.
SDq = D1 ** 0.5
NAMES = ['L2.A.knoi', 'L2.B.induction.diag', 'L2.C.state.diag']
LBL = {'L2.A.knoi': '$a_2$  noise veto', 'L2.B.induction.diag': '$b_2$  induction',
       'L2.C.state.diag': '$c_2$  state'}
COL = {'L2.A.knoi': '#2f6fc2', 'L2.B.induction.diag': '#c2492f', 'L2.C.state.diag': '#1a8f4a'}

fig, axes = plt.subplots(1, len(THRS), figsize=(4.3 * len(THRS), 4.2), sharey=True, squeeze=False)
print(f'{"alpha":>6} ' + ''.join(f'{LBL[k].split("  ")[1]:>14}' for k in NAMES)
      + f'{"b2 - c2":>12}{"final test CE":>15}')
for ax, thr in zip(axes[0], THRS):
    mt = json.load(open(f'{RUN}/snapmeta_x_{thr}.json'))
    gnames = mt['tie_group_names']['layer2.W_qk']
    _, _, rows = load_dof('layer2.W_qk', thr, 'w')
    kmax = mt['w_rows_written']
    steps = np.asarray(mt['w_steps'][:kmax])
    vals = {g: np.asarray(rows[:kmax, gnames.index(g)]) / SDq for g in NAMES}
    for g in NAMES:
        ax.plot(steps, vals[g], color=COL[g], lw=2, label=LBL[g])
    ax.axhline(0, color='0.7', lw=0.8)
    ax.set_title(f'$\\alpha$ = {thr}', fontsize=11)
    ax.set_xlabel('epoch'); ax.grid(alpha=0.25)
    *_, fin, _ = aligned_losses(thr)
    ce = 'still running' if np.isnan(fin) else f'{fin:.5f}'
    print(f'{thr:>6} ' + ''.join(f'{vals[g][-1]:>14.4f}' for g in NAMES)
          + f'{vals["L2.B.induction.diag"][-1] - vals["L2.C.state.diag"][-1]:>12.4f}{ce:>15}')
axes[0][0].set_ylabel('logits  (entry / $\\sqrt{108}$)')
axes[0][0].legend(fontsize=9.5)
fig.suptitle('Layer 2, in full: three numbers per threshold', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()